In [2]:
from pathlib import Path
import sys
import pandas as pd

cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "src").is_dir() else cwd.parent

if not (project_root / "src").is_dir():
    raise FileNotFoundError(f"Không tìm thấy thư mục src từ: {cwd}")

project_root_str = str(project_root)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

print(f"Project root: {project_root}")

Project root: D:\Intern\VCCorp\SwM_precomputed


In [3]:
from src.data.parser import read_behaviors, read_news

In [4]:
behaviors = read_behaviors(project_root / "data/raw/MINDsmall_train/behaviors.tsv")

In [5]:
behaviors.shape

(156965, 5)

In [30]:
behaviors.head()

,impression_id,user_id,time,history,impressions
0,1,U13740,11/11/2019 9:05:58 AM,N55189 N42782 N34694 N45794 N18445 N63302 N104...,N55689-1 N35729-0
1,2,U91836,11/12/2019 6:11:30 PM,N31739 N6072 N63045 N23979 N35656 N43353 N8129...,N20678-0 N39317-0 N58114-0 N20495-0 N42977-0 N...
2,3,U73700,11/14/2019 7:01:48 AM,N10732 N25792 N7563 N21087 N41087 N5445 N60384...,N50014-0 N23877-0 N35389-0 N49712-0 N16844-0 N...
3,4,U34670,11/11/2019 5:28:05 AM,N45729 N2203 N871 N53880 N41375 N43142 N33013 ...,N35729-0 N33632-0 N49685-1 N27581-0
4,5,U8125,11/12/2019 4:11:21 PM,N10078 N56514 N14904 N33740,N39985-0 N36050-0 N16096-0 N8400-1 N22407-0 N6...


In [6]:
unique_users = behaviors['user_id'].nunique()
unique_users

50000

In [7]:
len(behaviors[behaviors['user_id'].duplicated() == True])

106965

In [48]:
def have_consistent_history_lengths(
    df: pd.DataFrame,
    user_column: str = "user_id",
    history_column: str = "history",
) -> bool:
    """Return True when every row of each user has the same history length."""
    history_lengths = df[history_column].fillna("").str.split().str.len()
    unique_lengths_per_user = history_lengths.groupby(
        df[user_column], dropna=False
    ).nunique()

    return bool(unique_lengths_per_user.le(1).all())


same_lengths = pd.DataFrame(
    {"user_id": ["U1", "U1"], "history": ["N1 N2", "N3 N4"]}
)
different_lengths = pd.DataFrame(
    {"user_id": ["U1", "U1"], "history": ["N1 N2", "N3"]}
)
assert have_consistent_history_lengths(same_lengths)
assert not have_consistent_history_lengths(different_lengths)

result = have_consistent_history_lengths(behaviors)
result

True